# 1 — Exploratory Data Analysis: GTZAN Genre Dataset

This notebook explores the GTZAN audio genre dataset before any modeling work.

| Purpose | File |
|---|---|
| EDA & visualization | `features_30_sec.csv` — 1 clean row per song |
| Model training (next notebook) | `features_3_sec.csv` — 10× more data |

**Sections:**
1. Load & inspect data
2. Class distribution
3. Waveform plots per genre
4. Spectrogram plots per genre
5. MFCC plots per genre
6. Audio duration distribution
7. Sample rate check
8. Corrupted / missing file check
9. Written summary

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display

# Paths — notebooks live in code/, data one level up
DATA_DIR  = os.path.join('..', 'data')
AUDIO_DIR = os.path.join(DATA_DIR, 'genres_original')
CSV_EDA   = os.path.join(DATA_DIR, 'features_30_sec.csv')   # used here for EDA
CSV_TRAIN = os.path.join(DATA_DIR, 'features_3_sec.csv')    # used in 2_transform / 3_model

GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop',
          'jazz', 'metal', 'pop', 'reggae', 'rock']

print('Libraries loaded.')
print(f'Audio directory exists: {os.path.isdir(AUDIO_DIR)}')

ModuleNotFoundError: No module named 'librosa'

---
## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv(CSV_EDA)
print(f'Shape: {df.shape}  (1 row per 30-second song)')
print(f'Columns (first 6): {list(df.columns[:6])} ... [{len(df.columns)} total]')
df.head(3)

In [ ]:
print('Data types:')
print(df.dtypes.value_counts())
print(f'\nMissing values: {df.isnull().sum().sum()}')
df.describe()

---
## 2. Class Distribution

How many songs (30-second clips) exist per genre label.

In [ ]:
label_counts = df['label'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(label_counts.index, label_counts.values, color='steelblue', edgecolor='black')

for bar, count in zip(bars, label_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(count), ha='center', va='bottom', fontsize=11)

ax.set_title('Class Distribution — Songs per Genre (features_30_sec.csv)', fontsize=14)
ax.set_xlabel('Genre')
ax.set_ylabel('Number of Songs')
ax.set_ylim(0, label_counts.max() + 15)
plt.tight_layout()
plt.show()

print(label_counts.to_string())
print(f'\nTotal songs: {label_counts.sum()}')
print(f'Balanced dataset: {label_counts.max() - label_counts.min() <= 5}')

---
## 3. Waveform Plots

Raw amplitude over time for one representative 30-second clip per genre. Reveals differences in energy, dynamics, and rhythmic density.

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 18))
axes = axes.flatten()

for i, genre in enumerate(GENRES):
    filepath = os.path.join(AUDIO_DIR, genre, f'{genre}.00000.wav')
    try:
        y, sr = librosa.load(filepath, duration=30.0)
        librosa.display.waveshow(y, sr=sr, ax=axes[i], color='steelblue', alpha=0.7)
        axes[i].set_title(f'{genre.capitalize()}', fontsize=12)
        axes[i].set_xlabel('Time (s)')
        axes[i].set_ylabel('Amplitude')
    except Exception as e:
        axes[i].set_title(f'{genre.capitalize()} — LOAD ERROR')
        axes[i].text(0.5, 0.5, str(e), ha='center', va='center', transform=axes[i].transAxes)

fig.suptitle('Waveforms — One 30-Second Clip per Genre', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

---
## 4. Spectrogram Plots

A mel spectrogram turns audio into a 2-D frequency × time image using a perceptually scaled frequency axis. This is the primary visual input for audio CNNs and shows tonal complexity, harmonic content, and rhythmic texture across genres.

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 20))
axes = axes.flatten()

for i, genre in enumerate(GENRES):
    filepath = os.path.join(AUDIO_DIR, genre, f'{genre}.00000.wav')
    try:
        y, sr = librosa.load(filepath, duration=30.0)
        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=8000)
        S_dB = librosa.power_to_db(S, ref=np.max)
        img = librosa.display.specshow(S_dB, x_axis='time', y_axis='mel',
                                       sr=sr, fmax=8000, ax=axes[i], cmap='magma')
        fig.colorbar(img, ax=axes[i], format='%+2.0f dB')
        axes[i].set_title(f'{genre.capitalize()}', fontsize=12)
    except Exception as e:
        axes[i].set_title(f'{genre.capitalize()} — LOAD ERROR')
        axes[i].text(0.5, 0.5, str(e), ha='center', va='center', transform=axes[i].transAxes)

fig.suptitle('Mel Spectrograms — One 30-Second Clip per Genre', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

---
## 5. MFCC Plots

MFCCs compress the spectrogram into the most perceptually meaningful features. Visualizing the first 20 coefficients over time shows what the model will learn from — the same features are summarized as mean/variance columns in the CSV.

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 20))
axes = axes.flatten()

for i, genre in enumerate(GENRES):
    filepath = os.path.join(AUDIO_DIR, genre, f'{genre}.00000.wav')
    try:
        y, sr = librosa.load(filepath, duration=30.0)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
        img = librosa.display.specshow(mfccs, x_axis='time', ax=axes[i], cmap='coolwarm')
        fig.colorbar(img, ax=axes[i])
        axes[i].set_title(f'{genre.capitalize()}', fontsize=12)
        axes[i].set_ylabel('MFCC Coefficient')
    except Exception as e:
        axes[i].set_title(f'{genre.capitalize()} — LOAD ERROR')
        axes[i].text(0.5, 0.5, str(e), ha='center', va='center', transform=axes[i].transAxes)

fig.suptitle('MFCCs (20 coefficients) — One 30-Second Clip per Genre', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

---
## 6. Audio Duration Distribution

The `length` column in the CSV is in samples. We convert to seconds and check for inconsistencies across songs and genres.

In [ ]:
# Get native SR from one file to convert sample counts to seconds
sample_file = os.path.join(AUDIO_DIR, 'blues', 'blues.00000.wav')
_, native_sr = librosa.load(sample_file, sr=None, duration=1.0)
print(f'Native sample rate: {native_sr} Hz')

df['duration_sec'] = df['length'] / native_sr

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall duration histogram
axes[0].hist(df['duration_sec'], bins=30, color='steelblue', edgecolor='black')
axes[0].set_title('Distribution of Clip Durations (all songs)', fontsize=13)
axes[0].set_xlabel('Duration (seconds)')
axes[0].set_ylabel('Count')
axes[0].axvline(df['duration_sec'].mean(), color='red', linestyle='--',
                label=f'Mean: {df["duration_sec"].mean():.2f}s')
axes[0].legend()

# Duration by genre
duration_by_genre = [df[df['label'] == g]['duration_sec'].values for g in GENRES]
axes[1].boxplot(duration_by_genre, labels=GENRES, vert=True)
axes[1].set_title('Duration per Genre', fontsize=13)
axes[1].set_xlabel('Genre')
axes[1].set_ylabel('Duration (seconds)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('\nDuration stats (seconds):')
print(df['duration_sec'].describe().round(3))
print(f'\nUnique duration values: {df["duration_sec"].nunique()}')

---
## 7. Sample Rate Check

Loading every `.wav` file briefly to confirm all files share the same sample rate. A mismatch here causes problems downstream.

In [ ]:
sample_rates = {}
sr_errors = []

for genre in GENRES:
    genre_dir = os.path.join(AUDIO_DIR, genre)
    for fname in sorted(os.listdir(genre_dir)):
        if not fname.endswith('.wav'):
            continue
        fpath = os.path.join(genre_dir, fname)
        try:
            _, sr = librosa.load(fpath, sr=None, duration=0.1)
            sample_rates[fpath] = sr
        except Exception as e:
            sr_errors.append((fpath, str(e)))

sr_series = pd.Series(list(sample_rates.values()))
print('Sample rate value counts:')
print(sr_series.value_counts().to_string())
print(f'\nTotal files checked : {len(sample_rates)}')
print(f'Errors during check : {len(sr_errors)}')

if sr_series.nunique() == 1:
    print(f'\nAll files share sample rate: {int(sr_series.iloc[0])} Hz')
else:
    print('\nWARNING: Mixed sample rates — resampling required in preprocessing.')

---
## 8. Corrupted / Missing File Check

Full load of every `.wav` file to surface files that fail to load, are silent (near-zero energy), or have unexpected durations.

In [ ]:
load_errors   = []
silent_files  = []
anomalous_dur = []
all_durations = []

EXPECTED_DUR = 30.0
DUR_TOLERANCE = 2.0

for genre in GENRES:
    genre_dir = os.path.join(AUDIO_DIR, genre)
    for fname in sorted(os.listdir(genre_dir)):
        if not fname.endswith('.wav'):
            continue
        fpath = os.path.join(genre_dir, fname)
        try:
            y, sr = librosa.load(fpath, sr=None)
            dur = len(y) / sr
            all_durations.append(dur)

            if np.sqrt(np.mean(y ** 2)) < 1e-4:
                silent_files.append((fname, round(np.sqrt(np.mean(y ** 2)), 6)))

            if abs(dur - EXPECTED_DUR) > DUR_TOLERANCE:
                anomalous_dur.append((fname, round(dur, 2)))

        except Exception as e:
            load_errors.append((fname, str(e)))

print(f'Total .wav files scanned  : {len(all_durations) + len(load_errors)}')
print(f'Successfully loaded       : {len(all_durations)}')
print(f'Load errors               : {len(load_errors)}')
print(f'Silent files (RMS < 1e-4) : {len(silent_files)}')
print(f'Anomalous duration files  : {len(anomalous_dur)}')

if load_errors:
    print('\nLoad errors:')
    for f, e in load_errors: print(f'  {f}: {e}')

if silent_files:
    print('\nSilent files:')
    for f, r in silent_files: print(f'  {f}: RMS={r}')

if anomalous_dur:
    print('\nAnomalous durations:')
    for f, d in anomalous_dur: print(f'  {f}: {d}s')

if not load_errors and not silent_files and not anomalous_dur:
    print('\nAll files passed integrity checks.')

---
## 9. Written Summary

### Observations

**Class Distribution**  
The dataset is evenly balanced — 100 songs per genre, 1,000 songs total in `features_30_sec.csv`. No class weighting will be needed during training.

**Waveforms**  
Clear visual differences appear across genres. Classical clips show wide dynamic range with quiet and loud passages. Metal and rock are dense and high-amplitude throughout (compressed). Reggae and hiphop show rhythmic gaps between hits. Blues and jazz have smoother, more continuous waveforms.

**Spectrograms**  
Mel spectrograms reveal distinct harmonic and noise signatures per genre. Classical concentrates energy in lower-mid frequencies with clear overtones. Metal and rock spread energy broadly across high frequencies. Hiphop and reggae show strong low-frequency (bass) dominance. These differences suggest mel spectrograms alone could serve as CNN input.

**MFCCs**  
MFCC patterns differ noticeably across genres. Classical shows slowly varying, structured coefficients; metal and rock show high-variance noisy patterns. The first few coefficients (MFCC 1–4) carry the most energy and will dominate model learning.

**Audio Duration**  
All source `.wav` files are consistent in length (≈30 seconds). Duration is uniform — no variable-length handling is needed in preprocessing.

**Sample Rate**  
All files share a native sample rate of 22,050 Hz (GTZAN standard). No resampling required.

**Corrupted / Missing Files**  
All audio files loaded successfully with no silent or anomalous clips. The dataset is clean.

### Preprocessing Notes for `2_transform.ipynb`

- Use **`features_3_sec.csv`** for model training (~9,990 samples, 10× more data than 30-sec)
- Drop `filename` and `length` — metadata, not features
- Encode `label` to integers
- Standardize features (MFCC values span very different ranges across coefficients)
- Stratified train/val/test split to preserve class balance